# The Crab Nebula with H.E.S.S. public data — a first IACT analysis with Gammapy

**Level:** beginner · **Time:** ~3–4 h · **Gammapy:** ≥ 2.0 (tested with 2.1)

**Data:** H.E.S.S. DL3 DR1 (H.E.S.S. Collaboration 2018, [arXiv:1810.04516](https://arxiv.org/abs/1810.04516)), shipped in `$GAMMAPY_DATA/hess-dl3-dr1`
(the dataset repository formerly called *gammapy-extra*, now [gammapy-data](https://github.com/gammapy/gammapy-data)).
Background IRFs from Mohrmann et al. 2019, [A&A 632, A72](https://doi.org/10.1051/0004-6361/201936452).

Several coding "tricks" and the systematics study are adapted from the CADS 2025 beginners' hands-on
([B. Khélifi, *8. Systematic uncertainties*](https://github.com/bkhelifi/CADS_2025/tree/main/Beginners_handson)).

### What you will do
1. Explore the **data store** and select the Crab runs
2. Load the **observations**
3. Inspect **events and IRFs** and use them to choose the analysis binning
4. Build the **background** (background IRF + FoV normalisation)
5. Make **sky maps** (counts, background, significance)
6. **Simulate** the same observations with a known Crab model
7. Do a **1D spectral analysis** on the simulation and check you recover the truth
8. Do a **3D analysis** (spectral + spatial) on the simulation
9. Repeat on the **real data** and compare
10. Estimate the **systematic uncertainty** from the energy scale

Cells marked **✏️ TODO** contain `...` that you must replace. Questions marked **❓** are for discussion.
Lines marked `# TRICK` show useful Gammapy / Python features.

## 0. Setup

In [ ]:
import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord, Angle
from astropy.table import Table
from regions import CircleSkyRegion
from IPython.display import display

from gammapy.catalog import SourceCatalogGammaCat
from gammapy.data import DataStore
from gammapy.maps import MapAxis, WcsGeom, RegionGeom
from gammapy.datasets import MapDataset, SpectrumDataset, Datasets, MapDatasetEventSampler
from gammapy.makers import (
    DatasetsMaker, MapDatasetMaker, SafeMaskMaker, FoVBackgroundMaker,
    SpectrumDatasetMaker, ReflectedRegionsBackgroundMaker,
)
from gammapy.makers.utils import make_theta_squared_table
from gammapy.estimators import ExcessMapEstimator, FluxPointsEstimator
from gammapy.estimators.utils import resample_energy_edges
from gammapy.modeling import Fit, Parameter
from gammapy.modeling.selection import select_nested_models
from gammapy.modeling.models import (
    PowerLawSpectralModel, ExpCutoffPowerLawSpectralModel, LogParabolaSpectralModel, SpectralModel,
    PointSpatialModel, SkyModel, Models, FoVBackgroundModel, create_crab_spectral_model,
)
from gammapy.visualization import plot_spectrum_datasets_off_regions, plot_theta_squared_table
from gammapy.utils import pbar
import gammapy

pbar.SHOW_PROGRESS_BAR = True     # TRICK: progress bars for data reduction and flux points
warnings.filterwarnings("ignore")

N_JOBS = max(1, min(4, os.cpu_count() or 1))   # TRICK: number of CPU cores used for parallel steps

print("gammapy", gammapy.__version__)
print("GAMMAPY_DATA =", os.environ.get("GAMMAPY_DATA"))
print("N_JOBS =", N_JOBS)

In [ ]:
def print_params(model, truth=None, names=None):
    # Print fitted parameters as value ± error (and pull w.r.t. a 'truth' model if given)
    names = names or [param.name for param in model.parameters.free_parameters]
    for name in names:
        param = model.parameters[name]
        line = f"{name:10s} = {param.value:.4g} ± {param.error:.2g} {param.unit}"
        if truth is not None and name in truth.parameters.names:
            true_value = truth.parameters[name].quantity.to_value(param.unit)
            pull = (param.value - true_value) / param.error
            line += f"   (truth {true_value:.4g}, pull = {pull:+.2f})"
        print(line)


def obs_id_of(dataset):
    # Observation ID stored in the dataset metadata by the dataset makers
    return int(dataset.meta_table["OBS_ID"][0])


def sorted_by_obs_id(datasets):
    # Parallel reduction returns datasets in completion order: sort them by observation ID
    return Datasets(sorted(datasets, key=obs_id_of))


def rounded(table, decimals=2):
    # TRICK: round all numeric columns of a table for display
    table = table.copy()
    for colname in table.colnames:
        if table[colname].dtype.kind in "if":
            table[colname] = table[colname].round(decimals)
    return table

## 1. The data store

A `DataStore` is an index of DL3 files: an **observation index** (one row per run) and an **HDU index**
(where each events / IRF table lives). Format: [GADF](https://gamma-astro-data-formats.readthedocs.io).

In [ ]:
data_store = DataStore.from_dir("$GAMMAPY_DATA/hess-dl3-dr1")
data_store.info()

In [ ]:
obs_table = data_store.obs_table
print(obs_table.colnames)
print(np.unique(obs_table["OBJECT"]))

**✏️ TODO 1.** Select the runs pointed within 2° of the Crab with `obs_table.select_observations`
(selection `type="sky_circle"`), then plot the pointing positions.

In [ ]:
catalog = SourceCatalogGammaCat()
source = catalog["Crab"]
source

In [ ]:
crab_position = ...

selection = dict(type=..., frame="icrs",
                 lon=..., lat=..., radius=...)
crab_obs_table = obs_table.select_observations(...)
crab_obs_table["OBS_ID", "OBJECT", "RA_PNT", "DEC_PNT", "OFFSET_OBJ", "ZEN_PNT", "LIVETIME"].pprint()

pointing_geom = WcsGeom.create(skydir=crab_position, binsz=0.02, width=4 * u.deg, frame="icrs")
fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={"projection": pointing_geom.wcs})
ax.coords["ra"].set_format_unit(u.deg, decimal=True)
ax.coords["dec"].set_format_unit(u.deg, decimal=True)
pointings = SkyCoord(..., ..., unit="deg", frame="icrs")   # columns RA_PNT and DEC_PNT of crab_obs_table
ax.scatter(pointings.ra.deg, pointings.dec.deg, transform=ax.get_transform("icrs"), label="pointings")
ax.scatter(crab_position.ra.deg, crab_position.dec.deg, transform=ax.get_transform("icrs"),
           marker="*", s=200, color="r", label="Crab")
ax.legend()
plt.show()

❓ What observation strategy do you recognise in the pointing pattern? Why is it used?

## 2. Get the observations

In [ ]:
obs_ids = crab_obs_table["OBS_ID"]
observations = data_store.get_observations(obs_ids, required_irf="full-enclosure")
print(observations)
total_livetime = sum(obs.observation_live_time_duration for obs in observations)
print("Total livetime:", total_livetime.to("h"))

## 3. Events and IRFs → choosing the dataset axes

Before reducing data we must choose:
* the **reconstructed energy** axis (counts, background),
* the **true energy** axis (exposure, PSF, energy dispersion) — wider than the reco axis, to account for migration,
* the **spatial geometry** (pixel size, width) and the **maximum offset** used,
* the size of the **ON region** for the 1D analysis.

In [ ]:
first_obs = observations[0]
first_obs.events.peek()
plt.show()

### What are the IRFs?

An IACT does not measure photons directly: it records air showers and reconstructs an energy $E$ and a direction $p$
for each event. The **Instrument Response Functions (IRFs)** link these reconstructed quantities to the true ones,
$E_{\rm true}$ and $p_{\rm true}$. Following the
[Gammapy 2.1 user guide](https://docs.gammapy.org/2.1/user-guide/irf/index.html), the response factorises as

$$
R(p, E \mid p_{\rm true}, E_{\rm true}) = A_{\rm eff}(p_{\rm true}, E_{\rm true}) \times
{\rm PSF}(p \mid p_{\rm true}, E_{\rm true}) \times E_{\rm disp}(E \mid p_{\rm true}, E_{\rm true})
$$

and the expected number of events for an observation of livetime $t_{\rm obs}$ is the sum of a background
and a source contribution:

$$
N(p, E)\,{\rm d}p\,{\rm d}E = N_{\rm bkg}(p, E)\,{\rm d}p\,{\rm d}E + N_{\rm src}(p, E)\,{\rm d}p\,{\rm d}E
$$

$$
N_{\rm bkg}(p, E)\,{\rm d}p\,{\rm d}E = t_{\rm obs} \int_{E} {\rm d}E \int_{p} {\rm d}p\; {\rm Bkg}(p, E)
$$

$$
N_{\rm src}(p, E)\,{\rm d}p\,{\rm d}E = t_{\rm obs} \int_{E_{\rm true}} {\rm d}E_{\rm true} \int_{p_{\rm true}} {\rm d}p_{\rm true}\;
R(p, E \mid p_{\rm true}, E_{\rm true})\, \Phi(p_{\rm true}, E_{\rm true})
$$


Gammapy does this "forward folding" for you: you give a model $\Phi$ and it predicts counts in
**reconstructed** quantities, which are then compared with the data.

| IRF | Meaning | Unit | Depends on |
|---|---|---|---|
| `aeff` — effective area | collection area × detection & selection efficiency | m² | true energy, offset |
| `edisp` — energy dispersion | probability density of $\mu = E'/E$ | — | true energy, offset |
| `psf` — point spread function | probability density of the reconstructed direction around the true one | sr⁻¹ | true energy, offset |
| `bkg` — background | rate of gamma-like cosmic-ray events | s⁻¹ MeV⁻¹ sr⁻¹ | reco energy, position in the FoV |

The IRFs also depend on zenith angle, azimuth and the telescopes' optical efficiency. This is why they are
stored **per run** in DL3 files ([GADF format](https://gamma-astro-data-formats.readthedocs.io)).
Here they are *full-enclosure* IRFs: no θ cut was applied when deriving them, so the PSF is provided in full.

### How to read the `.peek()` plots

- **Effective area:** it rises steeply at low energy (threshold), reaches a plateau, and decreases with offset.
  The energy where it becomes small compared with the plateau is roughly where the analysis should start →
  this motivates the **safe energy threshold** (`LO_THRES` in the file header, used by `SafeMaskMaker("aeff-default")`).
- **Energy dispersion:** the migration matrix should be concentrated along $E'/E \simeq 1$.
  Its **width** is the energy resolution. A systematic shift from 1 is an energy **bias**, which is largest near threshold.
  Because events migrate across energy bins, the *true* energy axis must extend beyond the *reconstructed* one.
- **PSF:** read the 68 % containment radius at the energies you care about. It sets the useful pixel size
  (a few times smaller) and the size of the ON region for a point source.
- **Background:** its shape across the field of view depends on energy, so look at it energy by energy.
  Check up to which offset it is defined: this limits `offset_max`.

❓ Why does the background not need an energy dispersion?

In [ ]:
first_obs.aeff.peek(); plt.show()
first_obs.edisp.peek(); plt.show()
first_obs.psf.peek(); plt.show()
first_obs.bkg.peek(); plt.show()

The **θ² distribution** (squared angular distance of each event to the source, ON vs a mirrored OFF position)
shows directly whether the source is detected, and helps choosing the ON-region radius.

In [ ]:
on_region_radius = Angle("0.11 deg")

theta2_axis = MapAxis.from_bounds(0, 0.2, nbin=40, interp="lin", unit="deg2")
theta2_table = make_theta_squared_table(
    observations=observations, position=crab_position, theta_squared_axis=theta2_axis,
)
fig = plt.figure(figsize=(8, 5))
plot_theta_squared_table(theta2_table)
for ax in fig.get_axes():
    ax.axvline(on_region_radius.to_value("deg")**2, color="red", ls="--", label="ON region")
    ax.legend()
plt.show()

❓ Looking at the plots above:
* Where does the effective area become small?
* What is the 68 % containment radius of the PSF at 1 TeV? → choose a pixel size a few times smaller.
* Up to which offset is the background IRF defined?
* Where does the θ² excess vanish? Is 0.11° a sensible ON radius for a point source?

**✏️ TODO 3.** Define the analysis axes and geometry parameters.

In [ ]:
energy_axis = MapAxis.from_energy_bounds(..., ..., nbin=..., per_decade=True, unit="TeV", name="energy")
energy_axis_true = MapAxis.from_energy_bounds(..., ..., nbin=..., per_decade=True, unit="TeV", name="energy_true")
offset_max = ... * u.deg
pixel_size = ... * u.deg
map_width = ... * u.deg
print(energy_axis)
print(energy_axis_true)

## 4. Background

We use the **FoV background** method: the background IRF predicts the background in each pixel,
and its normalisation is fitted run by run on the region of the FoV **outside an exclusion mask**
around known sources (Berge, Funk & Hinton 2007, [A&A 466, 1219](https://doi.org/10.1051/0004-6361:20066674)).

`# TRICK`: for a bright source like the Crab, use an exclusion radius much larger than the ON region (here 0.5°),
so that the PSF tails do not leak into the background normalisation.

**✏️ TODO 4.** Create the geometry, the exclusion mask and the three makers.

In [ ]:
map_geom = WcsGeom.create(skydir=..., binsz=..., width=..., frame="icrs",
                          proj="CAR", axes=[...])

exclusion_region = CircleSkyRegion(center=..., radius=...)
exclusion_mask = ...   # True = usable for background (hint: region_mask on map_geom.to_image(), then invert with ~)
exclusion_mask.plot(); plt.show()

map_dataset_empty = MapDataset.create(geom=..., energy_axis_true=..., name="empty")
map_dataset_maker = MapDatasetMaker(selection=[...])   # the five maps needed for a 3D analysis
safe_mask_maker = SafeMaskMaker(methods=["aeff-max", "offset-max"], aeff_percent=..., offset_max=...)   # try 10 %
fov_background_maker = FoVBackgroundMaker(method="scale", exclusion_mask=...)

`# TRICK`: `DatasetsMaker` chains the makers **and runs the observations in parallel** (`n_jobs`).
The makers are applied in the order of the list, so the order matters (safe mask *before* FoV normalisation).

By default `DatasetsMaker` cuts each run around its pointing (width = 2 × `offset_max`). We give a larger
`cutout_width` so every run keeps the full map geometry and the runs can be compared and stacked directly.

We wrap the reduction in a function, because we will **reuse it unchanged on simulated observations**.

**✏️ TODO 4b.** Build the `DatasetsMaker` with the three makers in the right order.

In [ ]:
def reduce_3d(observations):
    # 3D data reduction: one MapDataset per run, sorted by observation ID
    makers = [...]   # the order matters
    datasets_maker = DatasetsMaker(..., stack_datasets=False, n_jobs=N_JOBS,
                                   cutout_width=2 * map_width)
    datasets = datasets_maker.run(..., ...)
    return sorted_by_obs_id(datasets)


map_datasets = reduce_3d(observations)
for dataset in map_datasets:
    print(obs_id_of(dataset), "background norm =",
          round(dataset.background_model.spectral_model.norm.value, 3), "$\pm",
          round(dataset.background_model.spectral_model.norm.error, 3))

❓ The fitted background norms are close to (but not exactly) 1. What could cause run-to-run deviations?

## 5. Maps

**✏️ TODO 5.** Stack the runs and compute a significance map with `ExcessMapEstimator`
(Li & Ma 1983, [ApJ 272, 317](https://doi.org/10.1086/161295)), using a 0.1° correlation radius.

In [ ]:
stacked_map_dataset = ...   # hint: Datasets.stack_reduce, name="stacked"
print(stacked_map_dataset)

excess_map_estimator = ExcessMapEstimator(correlation_radius=..., selection_optional=[])
significance_maps = ...

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), subplot_kw={"projection": map_geom.wcs})
stacked_map_dataset.counts.sum_over_axes().smooth(0.04 * u.deg).plot(ax=axes[0], add_cbar=True)
axes[0].set_title("counts")
stacked_map_dataset.background.sum_over_axes().plot(ax=axes[1], add_cbar=True)
axes[1].set_title("background")
significance_maps["sqrt_ts"].plot(ax=axes[2], add_cbar=True, cmap="coolwarm", vmin=-5, vmax=5)
axes[2].set_title("significance")
plt.show()

A good background model gives a **significance distribution outside the exclusion region compatible with N(0, 1)**.

In [ ]:
significance_values = significance_maps["sqrt_ts"].data
off_source_significance = significance_values[np.isfinite(significance_values) & exclusion_mask.data]
print(f"Off-source significance: mean = {np.mean(off_source_significance):.2f}, "
      f"std = {np.std(off_source_significance):.2f}  (expect ~0, ~1)")

fig, ax = plt.subplots()
ax.hist(off_source_significance, bins=50, range=(-5, 5), density=True, histtype="step", label="off-source pixels")
sigma_grid = np.linspace(-5, 5, 200)
ax.plot(sigma_grid, np.exp(-sigma_grid**2 / 2) / np.sqrt(2 * np.pi), label="N(0,1)")
ax.set_yscale("log"); ax.set_xlabel("significance"); ax.legend()
plt.show()

❓ Why is the width of this distribution not a perfect test (hint: neighbouring pixels share events when you correlate)?

## 6. Simulations

We now simulate the **same four runs** (same pointing, livetime and IRFs) with a known Crab model,
the H.E.S.S. exponential-cutoff power law of Aharonian et al. 2006,
[A&A 457, 899](https://doi.org/10.1051/0004-6361:20065351)
(available in Gammapy as `create_crab_spectral_model("hess_ecpl")`), plus the background IRF.

We sample **event lists** with `MapDatasetEventSampler`, so the simulated runs can be fed through **exactly the same
analysis** as the real ones.

In [ ]:
true_spectral_model = create_crab_spectral_model("hess_ecpl")
true_sky_model = SkyModel(
    spectral_model=true_spectral_model,
    spatial_model=PointSpatialModel(lon_0=crab_position.ra, lat_0=crab_position.dec, frame="icrs"),
    name="crab_true",
)
print(true_sky_model)

**✏️ TODO 6.** For each run: build a dataset with the IRFs (no counts), attach the true model + a background model,
sample events, and create a new `Observation` holding the simulated events.

Note: the event sampler needs the full energy-migration matrix, so we pass a `migra_axis` to `MapDataset.create`.

In [ ]:
simulation_maker = MapDatasetMaker(selection=["exposure", "background", "psf", "edisp"])
simulation_geom = WcsGeom.create(skydir=crab_position, binsz=pixel_size, width=6 * u.deg,
                                 frame="icrs", axes=[energy_axis])
migra_axis = MapAxis.from_bounds(0.2, 5, nbin=150, node_type="edges", name="migra")
simulation_dataset_empty = MapDataset.create(geom=simulation_geom, energy_axis_true=energy_axis_true,
                                             migra_axis=migra_axis, name="empty-sim")
event_sampler = MapDatasetEventSampler(random_state=42)

simulated_observations = []
for obs in observations:
    simulation_dataset = simulation_maker.run(simulation_dataset_empty.copy(name=f"sim-{obs.obs_id}"), ...)
    simulation_dataset.models = Models([..., FoVBackgroundModel(dataset_name=...)])
    simulated_events = event_sampler.run(..., ...)
    simulated_observations.append(obs.copy(in_memory=True, events=...))
    print(obs.obs_id, "simulated events:", len(simulated_events.table))

**✏️ TODO 6b.** Run the *same* 3D reduction on the simulated runs and compare the significance maps.

In [ ]:
sim_map_datasets = ...
sim_stacked_map_dataset = ...   # name="sim-stacked"
sim_significance_maps = ...
print("Counts in analysis geometry: real =", int(stacked_map_dataset.counts.data.sum()),
      " sim =", int(sim_stacked_map_dataset.counts.data.sum()))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), subplot_kw={"projection": map_geom.wcs})
for ax, maps, title in zip(axes, [significance_maps, sim_significance_maps], ["real", "simulated"]):
    maps["sqrt_ts"].plot(ax=ax, add_cbar=True, cmap="coolwarm", vmin=-5, vmax=30)
    ax.set_title(title)
plt.show()

## 7. 1D spectral analysis on the simulations

Classic ON/OFF analysis: a circular ON region (θ = 0.11°) and **reflected OFF regions**
(Berge, Funk & Hinton 2007), avoiding the exclusion mask. The fit uses the WSTAT statistic.

In [ ]:
on_region = CircleSkyRegion(center=crab_position, radius=on_region_radius)
spectrum_geom = RegionGeom.create(region=on_region, axes=[energy_axis])
spectrum_dataset_empty = SpectrumDataset.create(geom=spectrum_geom, energy_axis_true=energy_axis_true)
spectrum_dataset_maker = SpectrumDatasetMaker(selection=["counts", "exposure", "edisp"], containment_correction=True)
reflected_background_maker = ReflectedRegionsBackgroundMaker(exclusion_mask=exclusion_mask)
spectrum_safe_mask_maker = SafeMaskMaker(methods=["aeff-default"])

**✏️ TODO 7.** Complete the 1D reduction (again with `DatasetsMaker`) and the fit helper.

In [ ]:
def reduce_1d(observations):
    # 1D ON/OFF reduction: one SpectrumDatasetOnOff per run, sorted by observation ID
    makers = [...]
    datasets_maker = DatasetsMaker(..., stack_datasets=False, n_jobs=N_JOBS)
    datasets = datasets_maker.run(..., ...)
    # no models attached yet, so renaming by copy is safe: readable names in tables and plots
    return Datasets([dataset.copy(name=f"obs-{obs_id_of(dataset)}") for dataset in sorted_by_obs_id(datasets)])


def fit_spectrum(spectrum_datasets, spectral_model, name):
    # Stack the ON/OFF datasets and fit one spectral model
    stacked_dataset = ...   # name=f"{name}-stacked"
    stacked_dataset.models = SkyModel(spectral_model=..., name=name)
    fit_result = ...
    return stacked_dataset, fit_result


sim_spectrum_datasets = reduce_1d(simulated_observations)
sim_info_table = sim_spectrum_datasets.info_table(cumulative=True)
display(rounded(sim_info_table["livetime", "counts", "counts_off", "alpha", "excess", "sqrt_ts"]))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": exclusion_mask.geom.wcs})
exclusion_mask.plot(ax=ax)
on_region.to_pixel(ax.wcs).plot(ax=ax, edgecolor="k")
plot_spectrum_datasets_off_regions(ax=ax, datasets=sim_spectrum_datasets)
plt.show()

fig, (ax_excess, ax_sqrt_ts) = plt.subplots(1, 2, figsize=(10, 4))
livetime_hours = sim_info_table["livetime"].to("h")
ax_excess.plot(livetime_hours, sim_info_table["excess"], "o")
ax_excess.set(xlabel="Livetime [h]", ylabel="Excess events")
ax_sqrt_ts.plot(livetime_hours, sim_info_table["sqrt_ts"], "o")
ax_sqrt_ts.set(xlabel="Livetime [h]", ylabel="sqrt(TS)")
plt.show()

❓ How does the significance grow with livetime? What do you expect for a background-dominated vs signal-dominated regime?

We fit three spectral shapes and compare the fitted cutoff power law with the truth.

In [ ]:
def make_spectral_models():
    return {
        "pl": PowerLawSpectralModel(index=2.5, amplitude="3e-11 cm-2 s-1 TeV-1", reference="1 TeV"),
        "ecpl": ExpCutoffPowerLawSpectralModel(index=2.4, amplitude="3e-11 cm-2 s-1 TeV-1",
                                               reference="1 TeV", lambda_="0.05 TeV-1"),
        "lp": LogParabolaSpectralModel(alpha=2.4, beta=0.1, amplitude="3e-11 cm-2 s-1 TeV-1", reference="1 TeV"),
    }


sim_spectral_fits = {}
for model_key, spectral_model in make_spectral_models().items():
    sim_spectral_fits[model_key] = fit_spectrum(sim_spectrum_datasets, spectral_model, f"sim-{model_key}")
    fit_result = sim_spectral_fits[model_key][1]
    print(f"{model_key:5s} success={fit_result.success}  -2lnL={fit_result.total_stat:.1f}")

print("\nECPL fit on simulation vs truth:")
print_params(sim_spectral_fits["ecpl"][0].models[0].spectral_model, truth=true_spectral_model)

In [ ]:
sim_spectral_fits["ecpl"][0].plot_fit()
plt.show()

**Model comparison.** The power law (PL) is nested in the cutoff power law (ECPL, `lambda_ = 0`)
and in the log-parabola (LP, `beta = 0`). The likelihood-ratio test statistic
$TS = -2\ln L_{\rm null} + 2\ln L_{\rm alt}$ is asymptotically $\chi^2$-distributed with 1 d.o.f. (Wilks' theorem).

`# TRICK`: `select_nested_models` fits both hypotheses for you, and keeps the alternative model
only if it is preferred above `n_sigma`.

In [ ]:
def compare_nested(spectrum_datasets, alternative_key, null_parameter_name, n_sigma=3):
    # Likelihood-ratio test of PL (null) against ECPL or LP (alternative) on the stacked dataset
    stacked_dataset, _ = fit_spectrum(spectrum_datasets, make_spectral_models()[alternative_key], alternative_key)
    null_parameter = stacked_dataset.models[0].spectral_model.parameters[null_parameter_name]
    comparison = select_nested_models(Datasets([stacked_dataset]), parameters=[null_parameter],
                                      null_values=[0], n_sigma=n_sigma)
    return comparison["ts"]


for alternative_key, null_parameter_name in [("ecpl", "lambda_"), ("lp", "beta")]:
    ts = compare_nested(sim_spectrum_datasets, alternative_key, null_parameter_name)
    print(f"sim: PL -> {alternative_key:4s}  TS = {ts:5.1f}  (~{np.sqrt(max(ts, 0)):.1f} sigma, 1 dof)")

❓ Are the fitted parameters compatible with the truth (look at the pulls)? Is the cutoff significantly detected?
Change `random_state` in the event sampler and re-run: how much do the results move?

## 8. 3D analysis on the simulations

Now fit the stacked `MapDataset` with a point source (free position) + power law, together with
the FoV background model. The 3D fit uses the whole FoV and the PSF, rather than a single ON region.

**✏️ TODO 8.** Complete the 3D fit helper.

In [ ]:
def fit_3d(stacked_dataset, name):
    source_model = SkyModel(
        spectral_model=PowerLawSpectralModel(index=..., amplitude=..., reference="1 TeV"),
        spatial_model=PointSpatialModel(lon_0=..., lat_0=...,   # start ~0.05 deg away from the Crab
                                        frame="icrs"),
        name=name,
    )
    background_model = ...
    stacked_dataset.models = [...]
    return ...


sim_fit_result_3d = fit_3d(sim_stacked_map_dataset, "sim-3d")
print(sim_fit_result_3d)
print_params(sim_stacked_map_dataset.models["sim-3d"])

In [ ]:
sim_stacked_map_dataset.plot_residuals_spatial(method="diff/sqrt(model)", vmin=-0.5, vmax=0.5)  # smoothed 0.1 deg
plt.show()

❓ The truth is an ECPL but we fitted a PL. Compare the 3D PL index with the truth: is it closer to 2.39, or
steeper? Why? (Think about which energies dominate the statistics.)

## 9. Real data and comparison

**✏️ TODO 9.** Apply the *same* functions to the real observations.

In [ ]:
real_spectrum_datasets = ...

real_spectral_fits = {}
for model_key, spectral_model in make_spectral_models().items():
    real_spectral_fits[model_key] = ...   # name: f"real-{model_key}"
    fit_result = real_spectral_fits[model_key][1]
    print(f"{model_key:5s} success={fit_result.success}  -2lnL={fit_result.total_stat:.1f}")

real_stacked_map_dataset = stacked_map_dataset.copy(name="real-stacked")
real_fit_result_3d = ...   # name: "real-3d"
print(real_fit_result_3d)

In [ ]:
display(rounded(real_spectrum_datasets.info_table(cumulative=True)[
    "livetime", "counts", "counts_off", "alpha", "excess", "sqrt_ts"]))

In [ ]:
real_stacked_map_dataset.plot_residuals_spatial(method="diff/sqrt(model)", vmin=-0.5, vmax=0.5)
plt.show()

### 9a. Model comparison

In [ ]:
real_test_statistics = {}
for alternative_key, null_parameter_name in [("ecpl", "lambda_"), ("lp", "beta")]:
    ts = compare_nested(real_spectrum_datasets, alternative_key, null_parameter_name)
    real_test_statistics[alternative_key] = ts
    print(f"real: PL -> {alternative_key:4s}  TS = {ts:5.1f}  (~{np.sqrt(max(ts, 0)):.1f} sigma, 1 dof)")

### 9b. Parameters: real vs simulated, 1D vs 3D

In [ ]:
print("== 1D ECPL ==")
print("real:"); print_params(real_spectral_fits["ecpl"][0].models[0].spectral_model)
print("sim :"); print_params(sim_spectral_fits["ecpl"][0].models[0].spectral_model, truth=true_spectral_model)
print("\n== 1D PL ==")
print("real:"); print_params(real_spectral_fits["pl"][0].models[0].spectral_model)
print("sim :"); print_params(sim_spectral_fits["pl"][0].models[0].spectral_model)
print("\n== 3D PL + point source ==")
print("real:"); print_params(real_stacked_map_dataset.models["real-3d"])
print("sim :"); print_params(sim_stacked_map_dataset.models["sim-3d"])

### 9c. Spectral energy distribution

`# TRICK`: `resample_energy_edges` merges energy bins until each flux point reaches a minimum significance,
and `FluxPointsEstimator` can run on several cores (`n_jobs`).

**✏️ TODO 9c.** Compute flux points for real and simulated data and overlay them with the fitted and true models.

In [ ]:
sed_plot_kwargs = dict(energy_bounds=[0.7, 50] * u.TeV, sed_type="e2dnde")
sed_units = u.Unit("erg cm-2 s-1")   # TRICK: choose the plotting units with ax.yaxis.set_units(...)

flux_points_by_label = {}
fig, ax = plt.subplots(figsize=(7, 5))
ax.yaxis.set_units(sed_units)
for label, spectral_fits, color in [("real", real_spectral_fits, "C0"), ("sim", sim_spectral_fits, "C1")]:
    stacked_dataset = spectral_fits["ecpl"][0]
    energy_edges = resample_energy_edges(stacked_dataset, conditions=...)   # e.g. {"sqrt_ts_min": 2}
    flux_points_estimator = FluxPointsEstimator(
        energy_edges=..., source=...,
        selection_optional=["errn-errp", "ul"], n_jobs=N_JOBS,
    )
    flux_points = ...
    flux_points_by_label[label] = flux_points
    flux_points.plot(ax=ax, sed_type="e2dnde", color=color, label=f"{label} flux points")
    fitted_spectral_model = ...
    fitted_spectral_model.plot(ax=ax, color=color, ls="--", label=f"{label} ECPL fit", **sed_plot_kwargs)
    fitted_spectral_model.plot_error(ax=ax, facecolor=color, alpha=0.2, **sed_plot_kwargs)
true_spectral_model.plot(ax=ax, color="k", label="H.E.S.S. 2006 ECPL (sim. truth)", **sed_plot_kwargs)
ax.set_ylim(1e-12, 1e-10)
ax.set_ylabel(ax.get_ylabel().replace("e2dnde", "$E^2\\,dN/dE$"))   # TRICK: nicer axis label
ax.legend()
plt.show()

display(flux_points_by_label["real"].to_table(sed_type="e2dnde", formatted=True)[
    "e_ref", "e2dnde", "e2dnde_errn", "e2dnde_errp", "sqrt_ts", "success"])

## 10. Systematic uncertainty from the energy scale

The fit errors above are **statistical only**. A major IACT systematic is the **absolute energy scale**
(atmosphere, mirror reflectivity, ...). Suppose the reconstructed energies are biased by a factor $b$:
$E_{\rm reco} = b\,E$. Photons are conserved, so a true spectrum $\phi(E)$ is observed as

$$\phi_b(E) = \frac{1}{b}\,\phi\!\left(\frac{E}{b}\right).$$

For a power law $\phi \propto E^{-\Gamma}$, $\phi_b = b^{\,\Gamma-1}\phi$: the index is unchanged but the
normalisation moves by $(\Gamma-1)\,\delta b$; curved spectra also change shape.

**Monte Carlo recipe** (adapted from the CADS 2025 hands-on):
1. take the best-fit model on the **real** stacked ON/OFF dataset,
2. draw $b \sim \mathcal{N}(1, \sigma_E)$, fake ON/OFF counts from the *biased* model $\phi_b$,
3. refit with the *unbiased* model, store the parameters, repeat $N$ times,
4. repeat with $\sigma_E = 0$ (statistical fluctuations only).

Then $\sigma_{\rm syst} = \sqrt{\sigma_{\rm total}^2 - \sigma_{\rm stat}^2}$, with both widths taken from the MC.
We assume $\sigma_E = 10\,\%$ as in the CADS 2025 hands-on — change it and see what happens.

`# TRICK`: instead of writing one biased class per spectral shape, we write a single **wrapper** that
applies the energy bias to *any* spectral model (same pattern as Gammapy's `ScaleSpectralModel`).

In [ ]:
class EnergyScaleBiasedSpectralModel(SpectralModel):
    # phi_b(E) = phi(E / b) / b  for any wrapped spectral model phi
    tag = "EnergyScaleBiasedSpectralModel"
    bias = Parameter("bias", 1.0, frozen=True)

    def __init__(self, model, bias=1.0):
        self.model = model
        self._covariance = None
        super().__init__(bias=bias)

    def evaluate(self, energy, bias):
        return self.model(energy / bias) / bias


test_power_law = PowerLawSpectralModel(index=2.6)
biased_test_power_law = EnergyScaleBiasedSpectralModel(test_power_law, bias=1.1)
print("Check: ratio =", (biased_test_power_law(1 * u.TeV) / test_power_law(1 * u.TeV)).value,
      " expected b^(Gamma-1) =", 1.1 ** 1.6)

We use as reference the model preferred in **9a** (PL unless a curved model is preferred at > 3σ).

**✏️ TODO 10.** Complete the Monte Carlo loop.

In [ ]:
best_model_key = max(real_test_statistics, key=real_test_statistics.get)
if real_test_statistics[best_model_key] < 9:     # 3 sigma for 1 dof
    best_model_key = "pl"
print("Reference model for the systematics study:", best_model_key)

reference_dataset, reference_fit_result = real_spectral_fits[best_model_key]
best_spectral_model = reference_dataset.models[0].spectral_model
reference_npred_background = reference_dataset.npred_background()   # fixed once, outside the loop


def run_energy_bias_mc(energy_scale_sigma, n_simulations, seed):
    # Fake ON/OFF counts from the energy-biased best-fit model and refit with the unbiased model
    random_state = np.random.RandomState(seed)
    fitted_parameters = []
    for _ in range(n_simulations):
        bias = ...   # draw from N(1, energy_scale_sigma) with random_state
        fake_dataset = reference_dataset.copy(name="fake")   # always start from the real dataset
        fake_dataset.models = SkyModel(
            spectral_model=..., name="biased",   # the wrapper above, around best_spectral_model.copy()
        )
        fake_dataset.fake(npred_background=..., random_state=random_state)
        fake_dataset.models = SkyModel(spectral_model=..., name="unbiased")
        fit_result = ...
        if fit_result.success:
            row = {param.name: param.value for param in fit_result.parameters.free_parameters}
            row["bias"] = bias
            fitted_parameters.append(row)
    return Table(rows=fitted_parameters).to_pandas()   # TRICK: pandas for quick statistics


N_SIMULATIONS = 300
mc_stat_only = run_energy_bias_mc(energy_scale_sigma=0.0, n_simulations=N_SIMULATIONS, seed=1)
mc_with_bias = run_energy_bias_mc(energy_scale_sigma=0.10, n_simulations=N_SIMULATIONS, seed=2)
print(f"Successful fits: {len(mc_stat_only)} (stat only), {len(mc_with_bias)} (with energy bias)")

In [ ]:
for param in best_spectral_model.parameters.free_parameters:
    sigma_stat_hesse = param.error
    sigma_stat_mc = mc_stat_only[param.name].std()
    sigma_total_mc = mc_with_bias[param.name].std()
    sigma_syst = np.sqrt(max(sigma_total_mc**2 - sigma_stat_mc**2, 0))
    print(f"{param.name:10s} = {param.value:.3e} ± {sigma_stat_hesse:.2e} (stat, HESSE) "
          f"± {sigma_syst:.2e} (syst)  {param.unit}   [stat from MC: {sigma_stat_mc:.2e}]")

❓ Does the MC statistical width agree with the HESSE error? Which parameter is most affected by the energy scale, and does it match the formula above?

In [ ]:
corner_columns = [name for name in mc_with_bias.columns if name != "bias"]
corner_table = mc_with_bias[corner_columns].copy()
corner_table["amplitude"] *= 1e11          # 1e-11 cm-2 s-1 TeV-1 units for readability
try:
    import corner                          # optional dependency: pip install corner
    corner_labels = [name if name != "amplitude" else "amplitude [1e-11]" for name in corner_columns]
    corner.corner(corner_table.to_numpy(), labels=corner_labels,
                  quantiles=[0.16, 0.5, 0.84], show_titles=True)
except ImportError:
    import pandas as pd
    pd.plotting.scatter_matrix(corner_table, figsize=(10,10), diagonal="hist")
plt.show()

Finally the SED with the statistical band (from the fit) and the spread of models with the energy-scale uncertainty:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.yaxis.set_units(sed_units)
for _, mc_row in mc_with_bias.iterrows():
    mc_spectral_model = best_spectral_model.copy()
    for name in corner_columns:
        mc_spectral_model.parameters[name].value = mc_row[name]
    mc_spectral_model.plot(ax=ax, color="silver", alpha=0.15, zorder=1, **sed_plot_kwargs)

best_spectral_model.plot(ax=ax, color="C0", label=f"real, best fit ({best_model_key})", zorder=3, **sed_plot_kwargs)
best_spectral_model.plot_error(ax=ax, facecolor="C0", alpha=0.3, zorder=2, **sed_plot_kwargs)
flux_points_by_label["real"].plot(ax=ax, sed_type="e2dnde", color="C0", label="real flux points")
ax.plot([], [], color="silver", label="stat + energy-scale syst (MC)")
ax.set_ylim(1e-12, 1e-10)
ax.set_ylabel(ax.get_ylabel().replace("e2dnde", "$E^2\\,dN/dE$"))
ax.legend()
plt.show()

## Wrap-up questions ❓

1. Is the real Crab spectrum compatible with the H.E.S.S. 2006 reference once the energy-scale systematic is included?
2. With 1.75 h of data, can you significantly distinguish ECPL from LP? From PL? Compare with the same test on the simulation.
3. Is the 3D best-fit position compatible with the nominal Crab position given the statistical errors?
   Which systematic uncertainty would you add for an IACT?
4. The simulation uses the IRFs to *generate* events and then the same IRFs to *analyse* them. Which systematic
   uncertainties are therefore **absent** from the simulation but present in the real data?
5. Always check `FluxPoints` columns `success`, `sqrt_ts`, `counts`: a failed bin is silently set to NaN by default.

## Going further
* Repeat the simulation with 10 different random seeds and plot the distribution of the pulls (bias, width).
* Replace the point source by a `GaussianSpatialModel` and test the extension of the Crab with `select_nested_models`.
* Replace the 0.5° exclusion radius by 0.2° and 0.8°: how do the background norms and the excess change?
* Run the energy-scale MC with $\sigma_E$ = 5 %, 15 %, 20 %: how does the systematic scale?